In [ ]:
!pip install darts

In [ ]:
import os
import numpy as np
import pandas as pd

from darts import TimeSeries
from darts.dataprocessing import Pipeline
from darts.dataprocessing.transformers import Scaler, InvertibleMapper, StaticCovariatesTransformer
from darts.dataprocessing.transformers.missing_values_filler import MissingValuesFiller
from darts.metrics import rmsle
from darts.models import LinearRegressionModel, LightGBMModel, XGBModel, CatBoostModel
from darts.models.filtering.moving_average_filter import MovingAverageFilter
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from tqdm.notebook import tqdm_notebook

In [ ]:
PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting"

train       = pd.read_csv(os.path.join(PATH, "train.csv"),          parse_dates=["date"])
test        = pd.read_csv(os.path.join(PATH, "test.csv"),           parse_dates=["date"])
oil         = pd.read_csv(os.path.join(PATH, "oil.csv"),            parse_dates=["date"]).rename(columns={"dcoilwtico": "oil"})
store       = pd.read_csv(os.path.join(PATH, "stores.csv"))
transaction = pd.read_csv(os.path.join(PATH, "transactions.csv"),   parse_dates=["date"])
holiday     = pd.read_csv(os.path.join(PATH, "holidays_events.csv"),parse_dates=["date"])

train_start = train.date.min().date()
train_end   = train.date.max().date()


In [ ]:
# Fill missing Christmas dates
multi_idx = pd.MultiIndex.from_product(
    [pd.date_range(train_start, train_end), train.store_nbr.unique(), train.family.unique()],
    names=["date", "store_nbr", "family"],
)
train = train.set_index(["date", "store_nbr", "family"]).reindex(multi_idx).reset_index()
train[["sales", "onpromotion"]] = train[["sales", "onpromotion"]].fillna(0.)
train.id = train.id.interpolate(method="linear")

# Fill missing oil prices
oil = oil.merge(
    pd.DataFrame({"date": pd.date_range(train_start, test.date.max())}),
    on="date", how="outer",
).sort_values("date", ignore_index=True)
oil.oil = oil.oil.interpolate(method="linear", limit_direction="both")

# Fill missing transactions
store_sales = train.groupby(["date", "store_nbr"]).sales.sum().reset_index()
transaction = transaction.merge(store_sales, on=["date", "store_nbr"], how="outer").sort_values(
    ["date", "store_nbr"], ignore_index=True
)
transaction.loc[transaction.sales.eq(0), "transactions"] = 0.
transaction = transaction.drop(columns=["sales"])
transaction.transactions = transaction.groupby("store_nbr", group_keys=False).transactions.apply(
    lambda x: x.interpolate(method="linear", limit_direction="both")
)


In [ ]:
# Process holidays
def process_holiday(s):
    if "futbol" in s:
        return "futbol"
    to_remove = list(set(store.city.str.lower()) | set(store.state.str.lower()))
    for w in to_remove:
        s = s.replace(w, "")
    return s

holiday.description = holiday.apply(
    lambda x: x.description.lower().replace(x.locale_name.lower(), ""), axis=1,
).apply(process_holiday).replace(
    r"[+-]\d+|\b(de|del|traslado|recupero|puente|-)\b", "", regex=True,
).replace(r"\s+|-", " ", regex=True).str.strip()

holiday = holiday[holiday.transferred.eq(False)]

work_days = holiday[holiday.type.eq("Work Day")][["date", "type"]].rename(columns={"type": "work_day"}).reset_index(drop=True)
work_days.work_day = work_days.work_day.notna().astype(int)
holiday = holiday[holiday.type != "Work Day"].reset_index(drop=True)

local_holidays = holiday[holiday.locale.eq("Local")][["date", "locale_name", "description"]].rename(columns={"locale_name": "city"}).reset_index(drop=True)
local_holidays = local_holidays[~local_holidays.duplicated()]
local_holidays = pd.get_dummies(local_holidays, columns=["description"], prefix="loc")

regional_holidays = holiday[holiday.locale.eq("Regional")][["date", "locale_name", "description"]].rename(
    columns={"locale_name": "state", "description": "provincializacion"}
).reset_index(drop=True)
regional_holidays.provincializacion = regional_holidays.provincializacion.eq("provincializacion").astype(int)

national_holidays = holiday[holiday.locale.eq("National")][["date", "description"]].reset_index(drop=True)
national_holidays = national_holidays[~national_holidays.duplicated()]
national_holidays = pd.get_dummies(national_holidays, columns=["description"], prefix="nat")
national_holidays = national_holidays.groupby("date").sum().reset_index()
national_holidays = national_holidays.rename(columns={"nat_primer grito independencia": "nat_primer grito"})

In [ ]:
selected_holidays = [
    "nat_terremoto", "nat_navidad", "nat_dia la madre", "nat_dia trabajo",
    "nat_primer dia ano", "nat_futbol", "nat_dia difuntos",
]
keep_national_holidays = national_holidays[["date", *selected_holidays]]

data = pd.concat([train, test], axis=0, ignore_index=True).merge(
    transaction, on=["date", "store_nbr"], how="left",
).merge(oil, on="date", how="left").merge(store, on="store_nbr", how="left").merge(
    work_days, on="date", how="left",
).merge(keep_national_holidays, on="date", how="left").sort_values(
    ["date", "store_nbr", "family"], ignore_index=True
)

data[["work_day", *selected_holidays]] = data[["work_day", *selected_holidays]].fillna(0)
data["day"]         = data.date.dt.day
data["month"]       = data.date.dt.month
data["year"]        = data.date.dt.year
data["day_of_week"] = data.date.dt.dayofweek
data["day_of_year"] = data.date.dt.dayofyear
data["week_of_year"]= data.date.dt.isocalendar().week.astype(int)
data["date_index"]  = data.date.factorize()[0]

missing_dates = pd.date_range(train_start, train_end).difference(
    train.date.unique()
).strftime("%Y-%m-%d").tolist()
zero_sales_dates = missing_dates + [f"{j}-01-01" for j in range(2013, 2018)]
data.loc[(data.date.isin(zero_sales_dates)) & (data.sales.eq(0)) & (data.onpromotion.eq(0)), ["sales", "onpromotion"]] = np.nan

data.store_nbr = data.store_nbr.apply(lambda x: f"store_nbr_{x}")
data.cluster   = data.cluster.apply(lambda x: f"cluster_{x}")
data.type      = data.type.apply(lambda x: f"type_{x}")
data.city      = data.city.apply(lambda x: f"city_{x.lower()}")
data.state     = data.state.apply(lambda x: f"state_{x.lower()}")

In [ ]:
def get_pipeline(static_covs_transform=False, log_transform=False):
    lst = [MissingValuesFiller(n_jobs=-1)]
    if static_covs_transform:
        lst.append(StaticCovariatesTransformer(transformer_cat=OneHotEncoder(), n_jobs=-1))
    if log_transform:
        lst.append(InvertibleMapper(fn=np.log1p, inverse_fn=np.expm1, n_jobs=-1))
    lst.append(Scaler())
    return Pipeline(lst)


def get_target_series(static_cols, log_transform=True):
    target_dict, pipe_dict, id_dict = {}, {}, {}
    for fam in tqdm_notebook(data.family.unique(), desc="Extracting target series"):
        df   = data[(data.family.eq(fam)) & (data.date.le(train_end.strftime("%Y-%m-%d")))]
        pipe = get_pipeline(True, log_transform=log_transform)
        target = TimeSeries.from_group_dataframe(
            df=df, time_col="date", value_cols="sales",
            group_cols="store_nbr", static_cols=static_cols,
        )
        id_dict[fam]     = [{"store_nbr": t.static_covariates.store_nbr[0], "family": fam} for t in target]
        target           = pipe.fit_transform(target)
        target_dict[fam] = [t.astype(np.float32) for t in target]
        pipe_dict[fam]   = pipe[2:]
    return target_dict, pipe_dict, id_dict


def get_covariates(past_cols, future_cols, past_ma_cols=None, future_ma_cols=None,
                   past_window_sizes=[7, 28], future_window_sizes=[7, 28]):
    past_dict, future_dict = {}, {}
    covs_pipe = get_pipeline()

    for fam in tqdm_notebook(data.family.unique(), desc="Extracting covariates"):
        df = data[data.family.eq(fam)]

        past_covs = TimeSeries.from_group_dataframe(
            df=df[df.date.le(train_end.strftime("%Y-%m-%d"))],
            time_col="date", value_cols=past_cols, group_cols="store_nbr",
        )
        past_covs = [p.with_static_covariates(None) for p in past_covs]
        past_covs = covs_pipe.fit_transform(past_covs)
        if past_ma_cols:
            for size in past_window_sizes:
                ma = MovingAverageFilter(window=size)
                old = [f"rolling_mean_{size}_{c}" for c in past_ma_cols]
                new = [f"{c}_ma{size}" for c in past_ma_cols]
                past_covs = [p.stack(ma.filter(p[past_ma_cols]).with_columns_renamed(old, new)) for p in past_covs]
        past_dict[fam] = [p.astype(np.float32) for p in past_covs]

        future_covs = TimeSeries.from_group_dataframe(
            df=df, time_col="date", value_cols=future_cols, group_cols="store_nbr",
        )
        future_covs = [f.with_static_covariates(None) for f in future_covs]
        future_covs = covs_pipe.fit_transform(future_covs)
        if future_ma_cols:
            for size in future_window_sizes:
                ma = MovingAverageFilter(window=size)
                old = [f"rolling_mean_{size}_{c}" for c in future_ma_cols]
                new = [f"{c}_ma{size}" for c in future_ma_cols]
                future_covs = [f.stack(ma.filter(f[future_ma_cols]).with_columns_renamed(old, new)) for f in future_covs]
        future_dict[fam] = [f.astype(np.float32) for f in future_covs]

    return past_dict, future_dict


In [ ]:
static_cols = ["city", "state", "type", "cluster"]
target_dict, pipe_dict, id_dict = get_target_series(static_cols)

past_cols       = ["transactions"]
future_cols     = ["oil", "onpromotion", "day", "month", "year", "day_of_week",
                   "day_of_year", "week_of_year", "date_index", "work_day", *selected_holidays]
future_ma_cols  = ["oil", "onpromotion"]
past_dict, future_dict = get_covariates(past_cols, future_cols, future_ma_cols=future_ma_cols)


In [ ]:
TRAINER_CONFIG = {
    "target_dict": target_dict, "pipe_dict": pipe_dict, "id_dict": id_dict,
    "past_dict": past_dict,     "future_dict": future_dict,
    "forecast_horizon": 16, "folds": 1, "zero_fc_window": 21,
    "static_covs": "keep_all", "past_covs": "keep_all", "future_covs": "keep_all",
}


class Trainer:
    def __init__(self, target_dict, pipe_dict, id_dict, past_dict, future_dict,
                 forecast_horizon, folds, zero_fc_window,
                 static_covs=None, past_covs=None, future_covs=None):
        self.target_dict    = target_dict.copy()
        self.pipe_dict      = pipe_dict.copy()
        self.id_dict        = id_dict.copy()
        self.past_dict      = past_dict.copy()
        self.future_dict    = future_dict.copy()
        self.forecast_horizon = forecast_horizon
        self.folds          = folds
        self.zero_fc_window = zero_fc_window
        self.static_covs    = static_covs
        self.past_covs      = past_covs
        self.future_covs    = future_covs
        self.setup()

    def setup(self):
        for fam in tqdm_notebook(self.target_dict.keys(), desc="Setting up"):
            if self.static_covs != "keep_all":
                target = self.target_dict[fam]
                if self.static_covs is not None:
                    keep = [c for c in target[0].static_covariates.columns if c.startswith(tuple(self.static_covs))]
                    self.target_dict[fam] = [t.with_static_covariates(t.static_covariates[keep]) for t in target]
                else:
                    self.target_dict[fam] = [t.with_static_covariates(None) for t in target]
            if self.past_covs != "keep_all":
                self.past_dict[fam] = [p[self.past_covs] for p in self.past_dict[fam]] if self.past_covs else None
            if self.future_covs != "keep_all":
                self.future_dict[fam] = [p[self.future_covs] for p in self.future_dict[fam]] if self.future_covs else None

    def clip(self, arr):
        return np.clip(arr, a_min=0., a_max=None)

    def train_valid_split(self, target, length):
        train = [t[:-length] for t in target]
        end   = -length + self.forecast_horizon
        valid = [t[-length:(end if end < 0 else None)] for t in target]
        return train, valid

    def get_models(self, model_names, model_configs):
        registry = {"lr": LinearRegressionModel, "lgbm": LightGBMModel, "cat": CatBoostModel, "xgb": XGBModel}
        configs  = [c.copy() for c in model_configs]
        for i, name in enumerate(model_names):
            if name == "xgb":
                configs[i] = {"tree_method": "hist", **configs[i]}
        return [registry[name](**configs[j]) for j, name in enumerate(model_names)]

    def generate_forecasts(self, models, train, pipe, past_covs, future_covs, drop_before):
        if drop_before:
            date  = pd.Timestamp(drop_before) - pd.Timedelta(days=1)
            train = [t.drop_before(date) for t in train]
        inputs    = {"series": train, "past_covariates": past_covs, "future_covariates": future_covs}
        zero_pred = TimeSeries.from_dataframe(
            pd.DataFrame({"date": pd.date_range(train[0].end_time(), periods=self.forecast_horizon + 1)[1:],
                          "sales": np.zeros(self.forecast_horizon)}),
            time_col="date", value_cols="sales",
        )
        pred_list = []
        ens_pred  = [0] * len(train)
        for m in models:
            m.fit(**inputs)
            pred = m.predict(n=self.forecast_horizon, **inputs)
            pred = pipe.inverse_transform(pred)
            for j in range(len(train)):
                if train[j][-self.zero_fc_window:].values().sum() == 0:
                    pred[j] = zero_pred
            pred = [p.map(self.clip) for p in pred]
            pred_list.append(pred)
            for j in range(len(ens_pred)):
                ens_pred[j] += pred[j] / len(models)
        return pred_list, ens_pred

    def metric(self, valid, pred):
        return rmsle(valid, pred, inter_reduction=np.mean)

    def validate(self, model_names, model_configs, drop_before=None):
        pad = len(max(self.target_dict.keys(), key=len))
        model_metrics_history, ens_metric_history = [], []
        for fam in tqdm_notebook(self.target_dict, desc="Validating"):
            target, pipe = self.target_dict[fam], self.pipe_dict[fam]
            past_covs, future_covs = self.past_dict[fam], self.future_dict[fam]
            model_metrics, ens_metric = [], 0
            for j in range(self.folds):
                length = (self.folds - j) * self.forecast_horizon
                tr, valid = self.train_valid_split(target, length)
                valid     = pipe.inverse_transform(valid)
                ms        = self.get_models(model_names, model_configs)
                preds, ens = self.generate_forecasts(ms, tr, pipe, past_covs, future_covs, drop_before)
                model_metrics.append([self.metric(valid, p) / self.folds for p in preds])
                if len(ms) > 1:
                    ens_metric += self.metric(valid, ens) / self.folds
            model_metrics = np.sum(model_metrics, axis=0)
            model_metrics_history.append(model_metrics)
            ens_metric_history.append(ens_metric)
            print(fam, " " * (pad - len(fam)), " | ",
                  " - ".join([f"{n}: {v:.5f}" for n, v in zip(model_names, model_metrics)]),
                  (f" - ens: {ens_metric:.5f}" if len(ms) > 1 else ""), sep="")
        avg = np.mean(model_metrics_history, axis=0)
        print("Average RMSLE |", " - ".join([f"{n}: {v:.5f}" for n, v in zip(model_names, avg)]),
              f"- ens: {np.mean(ens_metric_history):.5f}" if len(ms) > 1 else "")

    def ensemble_predict(self, model_names, model_configs, drop_before=None):
        forecasts = []
        for fam in tqdm_notebook(self.target_dict.keys(), desc="Generating forecasts"):
            target, pipe = self.target_dict[fam], self.pipe_dict[fam]
            past_covs, future_covs = self.past_dict[fam], self.future_dict[fam]
            ms = self.get_models(model_names, model_configs)
            _, ens_pred = self.generate_forecasts(ms, target, pipe, past_covs, future_covs, drop_before)
            ens_pred = [p.to_dataframe().assign(**i) for p, i in zip(ens_pred, self.id_dict[fam])]
            forecasts.append(pd.concat(ens_pred, axis=0))
        forecasts = pd.concat(forecasts, axis=0).rename_axis(None, axis=1).reset_index(names="date")
        return forecasts



In [ ]:
BASE_CONFIG = {
    "random_state": 0,
    "lags": 63,
    "lags_past_covariates": list(range(-16, -23, -1)) if TRAINER_CONFIG["past_covs"] is not None else None,
    "lags_future_covariates": (14, 1) if TRAINER_CONFIG["future_covs"] is not None else None,
    "output_chunk_length": 1,
}

GBDT_CONFIG1 = {**BASE_CONFIG}
GBDT_CONFIG2 = {**BASE_CONFIG, "lags": 7}
GBDT_CONFIG3 = {**BASE_CONFIG, "lags": 365}
GBDT_CONFIG4 = {**BASE_CONFIG, "lags": 730}

ENS_MODELS  = ["lgbm", "lgbm", "lgbm", "lgbm"]
ENS_CONFIGS = [GBDT_CONFIG1, GBDT_CONFIG2, GBDT_CONFIG3, GBDT_CONFIG4]

trainer = Trainer(**TRAINER_CONFIG)

In [ ]:
predictions1 = trainer.ensemble_predict(model_names=ENS_MODELS, model_configs=ENS_CONFIGS)
predictions2 = trainer.ensemble_predict(model_names=ENS_MODELS, model_configs=ENS_CONFIGS, drop_before="2015-01-01")

final_predictions = predictions1.merge(predictions2, on=["date", "store_nbr", "family"], how="left")
final_predictions["sales"] = final_predictions[["sales_x", "sales_y"]].mean(axis=1)
final_predictions = final_predictions.drop(columns=["sales_x", "sales_y"])


def prepare_submission(predictions):
    predictions = predictions.copy()
    predictions.store_nbr = predictions.store_nbr.replace("store_nbr_", "", regex=True).astype(int)
    return test.merge(predictions, on=["date", "store_nbr", "family"], how="left")[["id", "sales"]]


submission = prepare_submission(final_predictions)
submission.to_csv("submission.csv", index=False)
print("Done! submission.csv saved.")

In [ ]:
!kaggle competitions submit -c store-sales-time-series-forecasting -f submission.csv  -m "Message"